In [ ]:
# Cell 0: Paths & Imports
from pathlib import Path
import numpy as np
import pandas as pd
from autogluon.tabular import TabularPredictor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

DATA = Path("/teamspace/studios/this_studio/data/dengue")
INTERIM = DATA/"interim"
FEATURES = DATA/"features"
REPORTS = DATA/"reports"
MODELS = DATA/"models"
for p in [FEATURES, REPORTS, MODELS]: p.mkdir(parents=True, exist_ok=True)

BASE_FE = FEATURES/"feature_matrix_weekly.parquet"
assert BASE_FE.exists(), f"Missing {BASE_FE}"
dm0 = pd.read_parquet(BASE_FE).copy()


In [ ]:
# Cell 1: Clean categorical keys + seasonal flags
dm = dm0.copy()
dm["week_start"] = pd.to_datetime(dm["week_start"])
if "district_id_txt_clean" not in dm.columns:
    dm["district_id_txt_clean"] = dm["district_id_txt"] if "district_id_txt" in dm.columns else dm["district"].astype(str)+"|"+dm["province"].astype(str)
dm["district_id_txt_clean"] = dm["district_id_txt_clean"].astype(str)
if "province_code" not in dm.columns:
    dm["province_code"] = dm["district_id_txt_clean"].str.split("|").str[-1]
dm["province_code"] = dm["province_code"].astype(str)
dm["district_num"] = pd.to_numeric(dm.get("district_num", np.nan), errors="coerce")
dm["district_num_cat"] = ("D"+dm["district_num"].fillna(-1).astype(int).astype(str)).astype(str)
dm["year"] = dm["week_start"].dt.year.astype(int)
dm["weekofyear"] = dm["week_start"].dt.isocalendar().week.astype(int)
dm["month"] = dm["week_start"].dt.month.astype(int)
dm["sin_woy"] = np.sin(2*np.pi*dm["weekofyear"]/52.0)
dm["cos_woy"] = np.cos(2*np.pi*dm["weekofyear"]/52.0)
dm["is_monsoon"] = dm["month"].between(5,10).astype(int)
dm["is_term1"] = dm["month"].between(5,9).astype(int)
dm["is_term2"] = dm["month"].isin([11,12,1,2,3]).astype(int)


In [ ]:
dm

In [ ]:
# Cell 2: Trend/enhanced temporal features per-district
def enrich_group(g):
    g = g.sort_values("week_start").copy()
    c = g["cases"].astype(float)
    lag1 = c.shift(1)
    g["cases_lag1"] = lag1
    g["cases_delta1"] = c - lag1
    g["cases_delta2"] = g["cases_delta1"] - g["cases_delta1"].shift(1)
    g["cases_ratio1"] = (c + 1.0) / (lag1 + 1.0)
    g["cases_logdiff1"] = np.log1p(c) - np.log1p(lag1)
    for W in [2,4,8]:
        g[f"cases_ewm{W}"] = c.ewm(span=W, adjust=False).mean().shift(1)
        g[f"cases_roll{W}_max"] = c.rolling(W, min_periods=max(1, W//2)).max().shift(1)
        g[f"cases_roll{W}_min"] = c.rolling(W, min_periods=max(1, W//2)).min().shift(1)
        g[f"cases_roll{W}_mean"] = c.rolling(W, min_periods=max(1, W//2)).mean().shift(1)
        g[f"cases_roll{W}_sum"] = c.rolling(W, min_periods=max(1, W//2)).sum().shift(1)
    wx_mean_cols = ["temperature_2m_mean","relative_humidity_2m_mean","vpd_kpa","windspeed_10m_max"]
    wx_sum_cols = ["precipitation_sum","shortwave_radiation_sum","rain_sum"]
    for W in [2,3,4,6,8]:
        for col in wx_mean_cols:
            if col in g:
                g[f"{col}_roll{W}_mean2"] = g[col].rolling(W, min_periods=max(1, W//2)).mean().shift(1)
        for col in wx_sum_cols:
            if col in g:
                g[f"{col}_roll{W}_sum2"] = g[col].rolling(W, min_periods=max(1, W//2)).sum().shift(1)
    if "precipitation_sum" in g:
        g["no_rain_flag"] = (g["precipitation_sum"].fillna(0.0) <= 0.001).astype(int)
        g["zero_rain_last3"] = g["no_rain_flag"].rolling(3, min_periods=1).sum().shift(1)
        g["zero_rain_last6"] = g["no_rain_flag"].rolling(6, min_periods=1).sum().shift(1)
    return g

dm = dm.groupby("district_id_txt_clean", group_keys=False).apply(enrich_group).reset_index(drop=True)
crit = ["cases_lag1"]
dm = dm.dropna(subset=[c for c in crit if c in dm.columns]).reset_index(drop=True)


In [ ]:
# Cell 3: Build horizons (h=1,h=2) targets
def build_horizon(df, h):
    def add_target(g):
        g = g.sort_values("week_start").copy()
        g["target_cases"] = g["cases"].shift(-h)
        g["target_log1p"] = np.log1p(g["target_cases"].astype(float))
        return g
    out = df.groupby("district_id_txt_clean", group_keys=False).apply(add_target).reset_index(drop=True)
    out = out.dropna(subset=["target_cases","target_log1p"]).reset_index(drop=True)
    return out

dm_h1 = build_horizon(dm, 1)
dm_h2 = build_horizon(dm, 2)


In [ ]:
# Cell 4: Time split per horizon (last 26 weeks as validation)
def time_split(df, last_weeks=26):
    uniq = df["week_start"].drop_duplicates().sort_values().to_list()
    cut = uniq[-last_weeks]
    train = df[df["week_start"] < cut].copy()
    valid = df[df["week_start"] >= cut].copy()
    return train, valid, cut

train_h1, valid_h1, cut_h1 = time_split(dm_h1, last_weeks=26)
train_h2, valid_h2, cut_h2 = time_split(dm_h2, last_weeks=26)

print("H1 valid from:", str(cut_h1.date()))
print("H2 valid from:", str(cut_h2.date()))


In [ ]:
train_h2

In [ ]:
valid_h2

In [ ]:
# Cell 5: Feature lists and categorical casting
def prepare_xy(df, label_key):
    drop_cols = {"week_start","target_cases","target_log1p"}
    if label_key == "target_cases":
        drop_cols.add("cases")
        drop_cols.add("cases_log1p")
    if label_key == "target_log1p":
        drop_cols.add("cases")
        drop_cols.add("cases_log1p")
    feats = [c for c in df.columns if c not in drop_cols]
    X = df[feats].copy()
    y = df[label_key].astype(float).copy()
    if "district_id_txt_clean" in X.columns:
        X["district_id_txt_clean"] = X["district_id_txt_clean"].astype("category")
    if "province_code" in X.columns:
        X["province_code"] = X["province_code"].astype("category")
    if "district_num_cat" in X.columns:
        X["district_num_cat"] = X["district_num_cat"].astype("category")
    return X, y, feats

def pack_train_valid(train_df, valid_df, label_key):
    Xtr, ytr, feats = prepare_xy(train_df, label_key)
    Xva, yva, _ = prepare_xy(valid_df, label_key)
    train_pack = Xtr.copy()
    train_pack[label_key] = ytr.values
    valid_pack = Xva.copy()
    valid_pack[label_key] = yva.values
    return train_pack, valid_pack, feats


In [ ]:
# Cell 6: Train function (AutoGluon, best_quality, 3000s) + evaluation
TIME_LIMIT = 3000

def train_and_eval(train_pack, valid_pack, label_key, model_tag):
    model_dir = MODELS / f"ag02_{model_tag}"
    predictor = TabularPredictor(
        label=label_key,
        problem_type="regression",
        eval_metric="root_mean_squared_error",
        path=str(model_dir)
    ).fit(
        train_data=train_pack,
        presets="best_quality",
        time_limit=TIME_LIMIT,
        verbosity=2
    )
    lb = predictor.leaderboard(valid_pack, silent=True)
    lb_path = REPORTS / f"leaderboard_{model_tag}.csv"
    lb.to_csv(lb_path, index=False)
    y_true = valid_pack[label_key].to_numpy()
    y_pred = predictor.predict(valid_pack.drop(columns=[label_key])).astype(float).to_numpy()
    return predictor, lb, lb_path, y_true, y_pred, model_dir

def metrics_on_cases(y_true_cases, y_pred_cases):
    mse = mean_squared_error(y_true_cases, y_pred_cases)
    rmse = float(np.sqrt(mse))
    mae = mean_absolute_error(y_true_cases, y_pred_cases)
    r2 = r2_score(y_true_cases, y_pred_cases)
    wape = (np.abs(y_pred_cases - y_true_cases).sum() / np.abs(y_true_cases).sum()) * 100 if y_true_cases.sum()!=0 else np.nan
    rmsle = float(np.sqrt(np.mean((np.log1p(np.maximum(0.0,y_pred_cases)) - np.log1p(np.maximum(0.0,y_true_cases)))**2)))
    return dict(RMSE=rmse, MAE=mae, R2=r2, WAPE=wape, RMSLE=rmsle)


In [ ]:
# Cell 7: H1 cases
train_pack_h1_cases, valid_pack_h1_cases, feats_h1_cases = pack_train_valid(train_h1, valid_h1, "target_cases")
pred_h1_cases, lb_h1_cases, lb_h1_cases_path, y_true_h1_cases, y_pred_h1_cases, dir_h1_cases = train_and_eval(
    train_pack_h1_cases, valid_pack_h1_cases, "target_cases", "h1_cases"
)
print("Saved leaderboard:", lb_h1_cases_path)
print(dir_h1_cases)
display(lb_h1_cases.head(20))
m_h1_cases = metrics_on_cases(y_true_h1_cases, np.maximum(0.0, y_pred_h1_cases))
print("[H1 | cases] Metrics:",
      f"\n - RMSE : {m_h1_cases['RMSE']:.4f}",
      f"\n - MAE  : {m_h1_cases['MAE']:.4f}",
      f"\n - R2   : {m_h1_cases['R2']:.4f}",
      f"\n - WAPE : {m_h1_cases['WAPE']:.2f}%",
      f"\n - RMSLE: {m_h1_cases['RMSLE']:.4f}")
pd.DataFrame({"y_true": y_true_h1_cases, "y_pred": np.maximum(0.0, y_pred_h1_cases)}).to_parquet(
    FEATURES/"ag02_valid_pred_h1_cases.parquet", index=False
)


In [ ]:
# Cell 8: H1 log1p
train_pack_h1_log, valid_pack_h1_log, feats_h1_log = pack_train_valid(train_h1, valid_h1, "target_log1p")
pred_h1_log, lb_h1_log, lb_h1_log_path, y_true_h1_log, y_pred_h1_log, dir_h1_log = train_and_eval(
    train_pack_h1_log, valid_pack_h1_log, "target_log1p", "h1_log1p"
)
print("Saved leaderboard:", lb_h1_log_path)
print(dir_h1_log)
display(lb_h1_log.head(20))
y_true_h1 = np.expm1(y_true_h1_log)
y_pred_h1 = np.maximum(0.0, np.expm1(y_pred_h1_log))
m_h1_log = metrics_on_cases(y_true_h1, y_pred_h1)
print("[H1 | log1p?cases] Metrics:",
      f"\n - RMSE : {m_h1_log['RMSE']:.4f}",
      f"\n - MAE  : {m_h1_log['MAE']:.4f}",
      f"\n - R2   : {m_h1_log['R2']:.4f}",
      f"\n - WAPE : {m_h1_log['WAPE']:.2f}%",
      f"\n - RMSLE: {m_h1_log['RMSLE']:.4f}")
pd.DataFrame({"y_true": y_true_h1, "y_pred": y_pred_h1}).to_parquet(
    FEATURES/"ag02_valid_pred_h1_log1p.parquet", index=False
)


In [ ]:
# Cell 9: H2 cases
train_pack_h2_cases, valid_pack_h2_cases, feats_h2_cases = pack_train_valid(train_h2, valid_h2, "target_cases")
pred_h2_cases, lb_h2_cases, lb_h2_cases_path, y_true_h2_cases, y_pred_h2_cases, dir_h2_cases = train_and_eval(
    train_pack_h2_cases, valid_pack_h2_cases, "target_cases", "h2_cases"
)
print("Saved leaderboard:", lb_h2_cases_path)
print(dir_h2_cases)
display(lb_h2_cases.head(20))
m_h2_cases = metrics_on_cases(y_true_h2_cases, np.maximum(0.0, y_pred_h2_cases))
print("[H2 | cases] Metrics:",
      f"\n - RMSE : {m_h2_cases['RMSE']:.4f}",
      f"\n - MAE  : {m_h2_cases['MAE']:.4f}",
      f"\n - R2   : {m_h2_cases['R2']:.4f}",
      f"\n - WAPE : {m_h2_cases['WAPE']:.2f}%",
      f"\n - RMSLE: {m_h2_cases['RMSLE']:.4f}")
pd.DataFrame({"y_true": y_true_h2_cases, "y_pred": np.maximum(0.0, y_pred_h2_cases)}).to_parquet(
    FEATURES/"ag02_valid_pred_h2_cases.parquet", index=False
)


In [ ]:
# Cell 10: H2 log1p
train_pack_h2_log, valid_pack_h2_log, feats_h2_log = pack_train_valid(train_h2, valid_h2, "target_log1p")
pred_h2_log, lb_h2_log, lb_h2_log_path, y_true_h2_log, y_pred_h2_log, dir_h2_log = train_and_eval(
    train_pack_h2_log, valid_pack_h2_log, "target_log1p", "h2_log1p"
)
print("Saved leaderboard:", lb_h2_log_path)
print(dir_h2_log)
display(lb_h2_log.head(20))
y_true_h2 = np.expm1(y_true_h2_log)
y_pred_h2 = np.maximum(0.0, np.expm1(y_pred_h2_log))
m_h2_log = metrics_on_cases(y_true_h2, y_pred_h2)
print("[H2 | log1p?cases] Metrics:",
      f"\n - RMSE : {m_h2_log['RMSE']:.4f}",
      f"\n - MAE  : {m_h2_log['MAE']:.4f}",
      f"\n - R2   : {m_h2_log['R2']:.4f}",
      f"\n - WAPE : {m_h2_log['WAPE']:.2f}%",
      f"\n - RMSLE: {m_h2_log['RMSLE']:.4f}")
pd.DataFrame({"y_true": y_true_h2, "y_pred": y_pred_h2}).to_parquet(
    FEATURES/"ag02_valid_pred_h2_log1p.parquet", index=False
)


In [ ]:
# Cell 11: Consolidated comparison table
rows = [
    {"run":"h1_cases", **m_h1_cases},
    {"run":"h1_log1p", **m_h1_log},
    {"run":"h2_cases", **m_h2_cases},
    {"run":"h2_log1p", **m_h2_log},
]
cmp = pd.DataFrame(rows)
cmp_path = REPORTS/"ag02_compare_h1_h2_cases_log1p.csv"
cmp.to_csv(cmp_path, index=False)
display(cmp)
print("Saved:", cmp_path)


In [ ]:
# Cell 12: Artifact index
index = {
    "leaderboards": {
        "h1_cases": str(REPORTS/"leaderboard_h1_cases.csv"),
        "h1_log1p": str(REPORTS/"leaderboard_h1_log1p.csv"),
        "h2_cases": str(REPORTS/"leaderboard_h2_cases.csv"),
        "h2_log1p": str(REPORTS/"leaderboard_h2_log1p.csv"),
    },
    "predictions": {
        "h1_cases": str(FEATURES/"ag02_valid_pred_h1_cases.parquet"),
        "h1_log1p": str(FEATURES/"ag02_valid_pred_h1_log1p.parquet"),
        "h2_cases": str(FEATURES/"ag02_valid_pred_h2_cases.parquet"),
        "h2_log1p": str(FEATURES/"ag02_valid_pred_h2_log1p.parquet"),
    },
    "comparison_table": str(REPORTS/"ag02_compare_h1_h2_cases_log1p.csv"),
}
print(index)
